### FAISS 구현 방식 비교: 생성자 vs 팩토리 메서드

| 구분 | 생성자 방식 (`FAISS()`) | 팩토리 메서드 방식 (`from_documents()`) |
| --- | --- | --- |
| **추상화 수준** | **저수준 (Low-level)** | **고수준 (High-level)** |
| **제어 권한** | 인덱스 종류, 거리 측정 방식 직접 지정 가능 | 기본 설정( 거리, IndexFlat) 사용 |
| **초기 데이터** | 데이터 없이 **빈 상태**로 시작 가능 | 초기 생성 시 **데이터 리스트** 필수 |
| **구현 난이도** | 차원 계산, 도큐먼트 스토어 결합 등 직접 수행 | 내부 로직에 의해 모든 과정 자동화 |
| **주요 용도** | 대규모 최적화, 커스텀 인덱스(HNSW 등) 필요 시 | 빠른 프로토타이핑, 표준적인 RAG 구현 |

### 핵심 차이점

#### 1. 인덱스 최적화 및 제어 (Control)

* **생성자 방식:** `faiss.IndexFlatL2`, `IndexIVFFlat`, `IndexHNSWFlat` 등 FAISS가 제공하는 다양한 인덱스 알고리즘을 개발자가 직접 선택할 수 있습니다. 이는 데이터 규모나 검색 속도 요구사항에 따른 정밀한 튜닝을 가능하게 합니다.
* **`from_documents`:** 가장 보편적인 `IndexFlatL2`(전수 조사 방식)로 자동 고정됩니다. 소규모 데이터에는 적합하나, 수백만 건 이상의 대규모 데이터 처리 시 성능 최적화에 제약이 있습니다.

#### 2. 초기화 프로세스의 자동화 (Automation)

* **생성자 방식:** 임베딩 모델의 차원(-dimension)을 미리 계산하고, `InMemoryDocstore`와 ID 매핑 테이블을 수동으로 결합해야 하는 번거로움이 있습니다.
* **`from_documents`:** 입력된 문서를 분석하여 임베딩 차원을 자동으로 파악하고, 내부적으로 인덱스와 저장소를 즉시 생성합니다. 코드가 간결하며 실수할 확률이 적습니다.

#### 3. 운영 유연성 (Flexibility)

* **생성자 방식:** 빈 인덱스를 먼저 생성해 둔 뒤, 애플리케이션 실행 중에 동적으로 데이터를 추가(`add_documents`)하는 구조에 유리합니다.
* **`from_documents`:** 인스턴스 생성과 데이터 적재가 동시에 일어나므로, 이미 준비된 정적 데이터를 한꺼번에 벡터화할 때 효율적입니다.

### 무엇을 선택해야 하는가?

* **엔지니어링 측면의 최적화가 중요하거나, 실시간으로 문서를 추가해야 하는 환경**이라면 **생성자 방식**을 사용하여 인덱스 구조를 직접 설계하는 것이 옳습니다.
* **빠른 기능 구현이 우선이며, 일반적인 문서 검색 성능으로도 충분한 상황**이라면 **`from_documents`** 메서드를 사용하여 코드 복잡도를 낮추는 것을 권장합니다.

> https://docs.langchain.com/oss/python/integrations/vectorstores#faiss

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# 1. 벡터 차원 정의: 사용 중인 임베딩 모델이 생성하는 결과물의 길이(차원)를 계산합니다.
# 임베딩 모델(embeddings)은 사전에 정의되어 있어야 합니다.
embedding_dim = len(embeddings.embed_query("hello world"))

# 2. FAISS 인덱스 초기화: 벡터 간의 거리 계산 방식을 결정합니다.
# IndexFlatL2: 유클리드 거리($L2$ distance)를 계산하는 가장 정확한 완전 탐색(Brute-force) 방식입니다.
index = faiss.IndexFlatL2(embedding_dim)

# 3. LangChain FAISS 객체 조립: 검색 엔진(index)과 원본 데이터 저장소(docstore)를 결합합니다.
vector_store = FAISS(
    embedding_function=embeddings,  # 텍스트를 벡터로 변환할 함수
    index=index,                   # 유사도 검색을 수행할 FAISS 인덱스
    docstore=InMemoryDocstore(),   # 실제 텍스트 내용과 메타데이터를 담을 메모리 저장소
    index_to_docstore_id={},       # 인덱스 번호와 저장소 ID 간의 매핑 테이블(초기화 시 빈 값)
)

# 4. 데이터 준비: 검색 대상이 될 문서 객체들을 생성합니다.
documents = [
    Document(page_content="컴퓨터 공학은 하드웨어와 소프트웨어를 연구하는 학문입니다.", metadata={"source": "edu"}),
    Document(page_content="인공지능은 데이터로부터 학습하여 지능적인 결정을 내리는 기술입니다.", metadata={"source": "tech"}),
    Document(page_content="고양이는 귀여운 동물이며 많은 사람들이 반려 동물로 키웁니다.", metadata={"source": "pets"}),
    Document(page_content="파이썬은 데이터 과학과 인공지능 분야에서 널리 쓰이는 언어입니다.", metadata={"source": "tech"}),
]

# 5. 데이터 적재: 문서를 임베딩하여 벡터화한 뒤 FAISS 인덱스에 추가합니다.
# 내부적으로 임베딩 생성 -> 인덱스 저장 -> docstore 저장이 동시에 수행됩니다.
vector_store.add_documents(documents=documents)

# 6. 유사도 검색(Top-K): 질문과 의미적으로 가장 가까운 문서 k개를 추출합니다.
query = "머신러닝과 AI 기술에 대해 알려줘"
results = vector_store.similarity_search(query, k=2)

print(f"--- [검색 질의]: {query} ---")
for i, doc in enumerate(results):
    print(f"결과 {i+1}: {doc.page_content} (출처: {doc.metadata['source']})")

# 7. 점수 포함 검색: 거리 값(Distance)을 포함하여 검색 결과의 신뢰도를 확인합니다.
# IndexFlatL2를 사용하므로 점수(Score)는 거리를 의미하며, 0에 가까울수록 유사도가 높습니다.
results_with_score = vector_store.similarity_search_with_score(query, k=4)

print("\n--- [점수 포함 검색 결과] ---")
for doc, score in results_with_score:
    print(f"거리(Score): {score:.4f} | 내용: {doc.page_content}")

--- [검색 질의]: 머신러닝과 AI 기술에 대해 알려줘 ---
결과 1: 인공지능은 데이터로부터 학습하여 지능적인 결정을 내리는 기술입니다. (출처: tech)
결과 2: 파이썬은 데이터 과학과 인공지능 분야에서 널리 쓰이는 언어입니다. (출처: tech)

--- [점수 포함 검색 결과] ---
거리(Score): 0.5265 | 내용: 인공지능은 데이터로부터 학습하여 지능적인 결정을 내리는 기술입니다.
거리(Score): 0.6992 | 내용: 파이썬은 데이터 과학과 인공지능 분야에서 널리 쓰이는 언어입니다.
거리(Score): 0.7824 | 내용: 컴퓨터 공학은 하드웨어와 소프트웨어를 연구하는 학문입니다.
거리(Score): 0.8486 | 내용: 고양이는 귀여운 동물이며 많은 사람들이 반려 동물로 키웁니다.


---